In [1]:
!pip install evaluate
!pip install rouge_score

  Using cached datasets-4.4.1-py3-none-any.whl.metadata (19 kB)
  Using cached dill-0.4.0-py3-none-any.whl.metadata (10 kB)
  Using cached multiprocess-0.70.18-py313-none-any.whl.metadata (7.2 kB)
  Using cached pyarrow-22.0.0-cp313-cp313-win_amd64.whl.metadata (3.3 kB)
Using cached datasets-4.4.1-py3-none-any.whl (511 kB)
Using cached dill-0.4.0-py3-none-any.whl (119 kB)
Using cached multiprocess-0.70.18-py313-none-any.whl (151 kB)
Using cached pyarrow-22.0.0-cp313-cp313-win_amd64.whl (28.0 MB)

   ---------------------------------------- 0/5 [pyarrow]
   ---------------------------------------- 0/5 [pyarrow]
   ---------------------------------------- 0/5 [pyarrow]
   ---------------------------------------- 0/5 [pyarrow]
   ---------------------------------------- 0/5 [pyarrow]
   ---------------------------------------- 0/5 [pyarrow]
   ---------------------------------------- 0/5 [pyarrow]
   ---------------------------------------- 0/5 [pyarrow]
   -------------------------------

In [3]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq, Seq2SeqTrainingArguments, Seq2SeqTrainer
from datasets import load_dataset
import torch

In [4]:
model = AutoModelForSeq2SeqLM.from_pretrained("t5-small")
tokenizer = AutoTokenizer.from_pretrained("t5-small")

d:\codes\summarizer\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\shahn\.cache\huggingface\hub\models--t5-small. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better perf

In [5]:
dataset_base = load_dataset("billsum")
train_dataset = dataset_base["train"].select(range(1000))
test_dataset = dataset_base["test"].select(range(200))
test_dataset

Dataset({
    features: ['text', 'summary', 'title'],
    num_rows: 200
})

In [74]:
MAX_OUT = 128
MAX_IN = 256
prefix = "summarize: "
input_text = prefix + dataset['test'][0]['text'] # Renamed 'input' to 'input_text' to avoid shadowing built-in 'input'
inputs = tokenizer(input_text, max_length=MAX_IN, truncation=True, return_tensors="pt").to(device)
# # Generate summary
summary_ids = model.generate(**inputs, max_length=MAX_OUT, min_length=64, length_penalty=2.0, num_beams=4, early_stopping=True)
res = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
res

"a) Jackson County, Mississippi.--Section 219 of the Water Resources Development Act of 1992 (106 Stat. 4835; 110 Stat. 3757) is amended by striking paragraph (5) and inserting the following: (5) Jackson county, mississippi.--Provision of an alternative water supply and a project for the elimination or control of combined sewer overflows for Jackson County, Mississippi.''"

In [79]:
def preprocess_function(examples):
    inputs = [prefix + doc for doc in examples["text"]]
    model_inputs = tokenizer(inputs, max_length=216, truncation=True)

    labels = tokenizer(text_target=examples["summary"], max_length=128, truncation=True)

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_train = train_dataset.map(preprocess_function, batched=True)
tokenized_test = test_dataset.map(preprocess_function, batched=True)

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    result = rouge.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)

    prediction_lens = [np.count_nonzero(pred != tokenizer.pad_token_id) for pred in predictions]
    result["gen_len"] = np.mean(prediction_lens)

    return {k: round(v, 4) for k, v in result.items()}

# 5. Training Arguments
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=checkpoint)

training_args = Seq2SeqTrainingArguments(
    output_dir="my_awesome_billsum_model",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=4, # Reduced batch size for memory
    per_device_eval_batch_size=4,
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=2,
    predict_with_generate=True,
    fp16=True, # Set to True if using CUDA
    push_to_hub=False,
    report_to="none", # Disable wandb reporting
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

/tmp/ipython-input-3179248430.py:44: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


In [82]:
# model performance before training
print("Model performance before training")
trainer.evaluate()

# model after training
trainer.train()
print("Model performance after training")
trainer.evaluate()

Model performance before training


Epoch,Training Loss,Validation Loss,Model Preparation Time,Rouge1,Rouge2,Rougel,Rougelsum,Gen Len
1,No log,2.987303,0.005600,0.187400,0.113200,0.164700,0.164800,20.000000
2,3.384000,2.925242,0.005600,0.200600,0.128900,0.181400,0.181400,20.000000


Model performance after training


{'eval_loss': 2.9252424240112305,
 'eval_model_preparation_time': 0.0056,
 'eval_rouge1': 0.2006,
 'eval_rouge2': 0.1289,
 'eval_rougeL': 0.1814,
 'eval_rougeLsum': 0.1814,
 'eval_gen_len': 20.0,
 'eval_runtime': 41.0467,
 'eval_samples_per_second': 4.872,
 'eval_steps_per_second': 1.218,
 'epoch': 2.0}

In [38]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

# Load the fine-tuned model and tokenizer
finetuned_model_path = "billsumodel"
loaded_tokenizer = AutoTokenizer.from_pretrained(finetuned_model_path)
loaded_model = AutoModelForSeq2SeqLM.from_pretrained(finetuned_model_path)

device = "cuda" if torch.cuda.is_available() else "cpu"
loaded_model.to(device)


T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [52]:
# Define a new sample text for summarization
sample_text_to_summarize = dataset['test'][1]['text'] # Using another text from the test set

# Prepare the input for the model
prefix = "summarize: "
input = prefix + sample_text_to_summarize

def preprocess(input, model, tokenizer):
# Tokenize the input
  input_ids = tokenizer(input, max_length=MAX_IN, truncation=True, return_tensors="pt").input_ids
  input_ids = input_ids.to(device)

  output_ids = model.generate(input_ids, max_length=MAX_OUT, num_beams=4, early_stopping=True)

  # Decode the generated summary
  generated_summary = loaded_tokenizer.decode(output_ids[0], skip_special_tokens=True)
  return generated_summary

# Generate the summary
generated_summary = preprocess(input, loaded_model, loaded_tokenizer)
print("Original Text:")
# print(sample_text_to_summarize)

print("\nGenerated Summary:")
print(generated_summary)



Original Text:

Generated Summary:
affidavit is effective upon receipt of the affidavit by the Secretary of State if the affidavit is received on or before the last day to register for an election to be held in the precinct of the person submitting the affidavit. for voter registration purposes, the applicant shall affirmatively assent to the use of his or her signature from his or her driver’s license or state identification card.


In [47]:
model = AutoModelForSeq2SeqLM.from_pretrained("t5-small")
tokenizer = AutoTokenizer.from_pretrained("t5-small")
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device) # Move the newly loaded model to the device
loaded_model.to(device)

orginalsummary = preprocess(input, model=model, tokenizer=tokenizer)
orginalsummary


'affidavit is effective upon receipt of the affidavit by the Secretary of State. affiant must affirmatively attest to the truth of the information provided in the affidavit.'

In [50]:
from huggingface_hub import notebook_login

notebook_login()

Once you've logged in, you can push your fine-tuned model to the Hugging Face Hub. Replace `"your-username/your-model-name"` with your desired repository name.

In [83]:
# Push the fine-tuned model and tokenizer to the Hugging Face Hub
# Replace "your-username/your-model-name" with your desired repository name
repo_name = "my-awesome-billsum-model"
# loaded_tokenizer.push_to_hub(repo_name)
# loaded_model.push_to_hub(repo_name)
trainer.push_to_hub(repo_name)

print(f"Model and tokenizer pushed to https://huggingface.co/{repo_name}")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...l_summarizer/spiece.model: 100%|##########|  792kB /  792kB            

  ...marizer/model.safetensors:   0%|          |  551kB /  242MB            

  ...marizer/training_args.bin:   1%|1         |  87.0B / 5.97kB            

Model and tokenizer pushed to https://huggingface.co/my-awesome-billsum-model


In [84]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("SlimeRimuru/billsum_model_summarizer")
model = AutoModelForSeq2SeqLM.from_pretrained("SlimeRimuru/billsum_model_summarizer")


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/122 [00:00<?, ?B/s]